#### Imports

In [84]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from datetime import datetime, timedelta
import warnings
from typing import Dict, Tuple
from pathlib import Path
from scipy.stats import norm, invgamma, multivariate_normal
import pickle
import os

from scipy import stats as scipy_stats
from scipy.stats import skewnorm, t, nct, norminvgauss
from scipy.optimize import minimize
from skewt_scipy.skewt import skewt

from joblib import Parallel, delayed
import multiprocessing

warnings.filterwarnings('ignore')
# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

import contextlib
import io

from kama_msr import KAMA
from kama_msr import MarkovSwitchingModel
from kama_msr import KAMA_MSR
from kmrf import KMRF
from KMRF_training_config import *
from bayesian_forward_simulator import BayesianForwardSimulator
from portfolio_optimizer_inputs import PortfolioOptimizerInputs
from portfolio_optimizer import PortfolioOptimizer

---
## Data

#### --------------------------------------------------------------------

In [85]:
# Prepare data: etf_close_prices, commodity_close_prices, universe_close_prices
international_index_symbol_names = pd.read_csv('data/inputs/fmp_index_list.csv').set_index('symbol')['name']
international_index_symbol_names = international_index_symbol_names[~international_index_symbol_names.index.isin(['^GSPC', '^NDX'])].to_dict()
commodity_symbol_names = pd.read_csv('data/inputs/fmp_commodity_list.csv').set_index('symbol')['name'].to_dict()
etf_symbol_names = {
    # BOND ETFS
    'BIL': 'SPDR Bloomberg 1-3 Month T-Bill ETF',
    'SHY': 'iShares 1-3 Year Treasury Bond ETF',
    'IEF': 'iShares 7-10 Year Treasury Bond ETF',
    # International EQUITY ETFS
    'VXUS': 'Vanguard Total International Stock ETF',
    'VEA': 'Vanguard FTSE Developed Markets ETF',
    'VWO': 'Vanguard FTSE Emerging Markets ETF',
    'VGK': 'Vanguard FTSE Europe ETF',
    'VPL': 'Vanguard FTSE Pacific ETF',
    'FXI': 'iShares China Large-Cap ETF',
    'EWJ': 'iShares MSCI Japan ETF',
    'INDA': 'iShares MSCI India ETF',
    # MAJOR INDICES
    '^GSPC': 'S&P 500',
    '^IXIC': 'Nasdaq Composite',
    '^NDX': 'Nasdaq 100',
    '^RUT': 'Russell 2000',
    '^DJI': 'Dow Jones Industrial Average',
    '^RUI': 'Russell 1000',
    '^RUA': 'Russell 3000',
    
    # MAIN BROAD MARKET ETFS
    'SPY': 'SPDR S&P 500 ETF',
    'VOO': 'Vanguard S&P 500 ETF',
    'RSP': 'Invesco S&P 500 Equal Weight ETF',
    'IVV': 'iShares Core S&P 500 ETF',
    'QQQ': 'Invesco QQQ Trust',
    'QQQM': 'Invesco Nasdaq 100 ETF',
    'ONEQ': 'Fidelity Nasdaq Composite Index ETF',
    'IWM': 'iShares Russell 2000 ETF',
    'IWB': 'iShares Russell 1000 ETF',
    'IWV': 'iShares Russell 3000 ETF',
    'DIA': 'SPDR Dow Jones Industrial Average ETF',
    'VTI': 'Vanguard Total Stock Market ETF',
    
    # S&P 500 SECTOR ETFS (SELECT SECTOR SPDRS)
    'XLE': 'Energy Select Sector SPDR',
    'XLF': 'Financial Select Sector SPDR',
    'XLU': 'Utilities Select Sector SPDR',
    'XLI': 'Industrial Select Sector SPDR',
    'XLV': 'Health Care Select Sector SPDR',
    'XLK': 'Technology Select Sector SPDR',
    'XLB': 'Materials Select Sector SPDR',
    'XLY': 'Consumer Discretionary Select Sector SPDR',
    'XLP': 'Consumer Staples Select Sector SPDR',
    'XLRE': 'Real Estate Select Sector SPDR',
    'XLC': 'Communication Services Select Sector SPDR',
    
    # GROWTH ETFs
    'IVW': 'iShares S&P 500 Growth ETF',
    'VONG': 'Vanguard Russell 1000 Growth ETF',
    'IWF': 'iShares Russell 1000 Growth ETF',
    'IWO': 'iShares Russell 2000 Growth ETF',
    'VUG': 'Vanguard Growth ETF',
    'SPYG': 'SPDR Portfolio S&P 500 Growth ETF',
    
    # VALUE ETFs
    'IVE': 'iShares S&P 500 Value ETF',
    'VONV': 'Vanguard Russell 1000 Value ETF',
    'IWD': 'iShares Russell 1000 Value ETF',
    'IWN': 'iShares Russell 2000 Value ETF',
    'VTV': 'Vanguard Value ETF',
    'SPYV': 'SPDR Portfolio S&P 500 Value ETF',
    
    # SIZE ETFs
    'IWR': 'iShares Russell Mid-Cap ETF',
    'IWC': 'iShares Micro-Cap ETF',
    'IJH': 'iShares Core S&P Mid-Cap ETF',
    'IJR': 'iShares Core S&P Small-Cap ETF',
    'MDY': 'SPDR S&P MidCap 400 ETF',
    'SLY': 'SPDR S&P 600 Small Cap ETF',
    'VO': 'Vanguard Mid-Cap ETF',
    'VB': 'Vanguard Small-Cap ETF',
    'SCHA': 'Schwab U.S. Small-Cap ETF',
    'SCHM': 'Schwab U.S. Mid-Cap ETF',
    'VTWO': 'Vanguard Russell 2000 ETF',
    'VTHR': 'Vanguard Russell 3000 ETF',
    'THRK': 'iShares Russell 3000 ETF',
    'SPSM': 'SPDR Portfolio S&P 600 Small Cap ETF',
    'SMLF': 'iShares Small-Cap US Equity Factor ETF',
    
    # NASDAQ SPECIFIC
    'QTEC': 'First Trust Nasdaq-100 Technology Sector Index Fund',
    'QQEW': 'First Trust Nasdaq-100 Equal Weighted Index Fund',
    'QQQG': 'Pacer Nasdaq 100 Top 50 Cash Cows Dividend Growth ETF',
    'QQQV': 'Pacer Nasdaq 100 Top 50 Value ETF',
    
    # DIVIDEND/QUALITY
    'SCHD': 'Schwab U.S. Dividend Equity ETF',
    'VYM': 'Vanguard High Dividend Yield ETF',
    'DVY': 'iShares Select Dividend ETF',
    'QUAL': 'iShares MSCI USA Quality Factor ETF',
    'USMV': 'iShares MSCI USA Min Vol Factor ETF',
    
    # EQUAL WEIGHT
    'EWSC': 'Invesco S&P SmallCap 600 Equal Weight ETF',
    'EWMC': 'Invesco S&P MidCap 400 Equal Weight ETF',
}
universe_symbol_names = {
    'IVV': 'IVV - iShares Core S&P 500 ETF',
    'IJH': 'IJH - iShares Core S&P Mid-Cap ETF',
    'IWM': 'IWM - iShares Russell 2000 ETF',
    'EFA': 'EFA - iShares MSCI EAFE ETF',
    'EEM': 'EEM - iShares MSCI Emerging Markets ETF',
    'AGG': 'AGG - iShares Core U.S. Aggregate Bond ETF',
    'SPTL': 'SPTL - SPDR Portfolio Long Term Treasury ETF',
    'HYG': 'HYG - iShares iBoxx $ High Yield Corporate Bond ETF',
    'SPBO': 'SPBO - SPDR Portfolio Corporate Bond ETF',
    'IYR': 'IYR - iShares U.S. Real Estate ETF',
    'DBC': 'DBC - Invesco DB Commodity Index Tracking Fund',
    'GLD': 'GLD - SPDR Gold Shares',
}

# international_index_data = pd.read_csv('data/processed/index_data.csv', index_col=0, header=[0, 1], parse_dates=True)
commodity_data = pd.read_csv('data/processed/commodity_data.csv', index_col=0, header=[0, 1], parse_dates=True)
etf_data = pd.read_csv('data/processed/all_etf_data.csv', index_col=0, header=[0, 1], parse_dates=True)
universe_data = pd.read_csv('data/processed/universe_etfs.csv', index_col=0, header=[0, 1], parse_dates=True)

commodity_data_close_cols = commodity_data.columns[commodity_data.columns.get_level_values(1) == 'close']
commodity_close_prices = commodity_data[commodity_data_close_cols].droplevel(1, axis=1).rename(columns=commodity_symbol_names)
commodity_close_prices.columns = [col.replace('/', ' ') for col in commodity_close_prices.columns]

etf_close_cols = etf_data.columns[etf_data.columns.get_level_values(1) == 'close']
etf_close_prices = etf_data[etf_close_cols].droplevel(1, axis=1).rename(columns=etf_symbol_names)

universe_close_cols = universe_data.columns[universe_data.columns.get_level_values(1) == 'close']
universe_close_prices = universe_data[universe_close_cols].droplevel(1, axis=1).rename(columns=universe_symbol_names)

In [86]:
etf_always_disclude = ['Vanguard S&P 500 ETF', 'Real Estate Select Sector SPDR', 'Communication Services Select Sector SPDR']
etf_disclude = [name for name in etf_data.columns if 'Russell 1000' in name or 'Russell 3000' in name]\
                        + ['S&P 500', 'Nasdaq Composite', 'Dow Jones Industrial Average', 'Nasdaq 100', 'Russell 2000']

etf_include = list(set(etf_close_prices.columns.tolist()) - set(etf_always_disclude) - set(etf_disclude))
etf_close_prices = etf_close_prices[etf_include]

us_treasury = ['SPDR Bloomberg 1-3 Month T-Bill ETF', 'iShares 1-3 Year Treasury Bond ETF', 'iShares 7-10 Year Treasury Bond ETF']
int_equity = ['Vanguard Total International Stock ETF', 'Vanguard FTSE Developed Markets ETF',\
                                        'Vanguard FTSE Emerging Markets ETF','Vanguard FTSE Europe ETF',\
                                        'Vanguard FTSE Pacific ETF', 'iShares China Large-Cap ETF',\
                                        'iShares MSCI Japan ETF', 'iShares MSCI India ETF']
us_equity = list(set(etf_include) - set(us_treasury) - set(int_equity))

#### --------------------------------------------------------------------

In [87]:
all_rebal_dates = etf_close_prices['SPDR S&P 500 ETF'].to_frame().dropna().loc['2018-12-31':].index[::21].map(lambda dte: dte.strftime('%Y%m%d')).to_list()
all_dates = etf_close_prices.loc[all_rebal_dates[0]:].index[1:]

In [88]:
# Create asset_names_df
from KMRF_training_config import *
asset_names_df = pd.DataFrame({
    'universe': get_assets_by_class('universe') + ['']*7,
    'us_equity': get_assets_by_class('us_equity'),
    'commodity': get_assets_by_class('commodity') + ['']*5,
    'int_equity': get_assets_by_class('int_equity') + ['']*11,

})

from pandas import option_context
with option_context('display.max_colwidth', None):
    display(asset_names_df)

,universe,us_equity,commodity,int_equity
0,IVV - iShares Core S&P 500 ETF,SPDR S&P 500 ETF,Gold Futures,Vanguard Total International Stock ETF
1,IJH - iShares Core S&P Mid-Cap ETF,Invesco QQQ Trust,Wheat Futures,Vanguard FTSE Developed Markets ETF
2,IWM - iShares Russell 2000 ETF,iShares Russell 2000 ETF,Corn Futures,Vanguard FTSE Emerging Markets ETF
3,EFA - iShares MSCI EAFE ETF,SPDR Dow Jones Industrial Average ETF,Copper,Vanguard FTSE Europe ETF
4,EEM - iShares MSCI Emerging Markets ETF,Energy Select Sector SPDR,Sugar,Vanguard FTSE Pacific ETF
5,AGG - iShares Core U.S. Aggregate Bond ETF,Financial Select Sector SPDR,Silver Futures,iShares China Large-Cap ETF
6,SPTL - SPDR Portfolio Long Term Treasury ETF,Utilities Select Sector SPDR,US Dollar,iShares MSCI Japan ETF
7,HYG - iShares iBoxx $ High Yield Corporate Bond ETF,Industrial Select Sector SPDR,Soybean Futures,iShares MSCI India ETF
8,SPBO - SPDR Portfolio Corporate Bond ETF,Health Care Select Sector SPDR,Lumber Futures,
9,IYR - iShares U.S. Real Estate ETF,Technology Select Sector SPDR,Live Cattle Futures,


#### --------------------------------------------------------------------

In [89]:
etf_close_prices[us_equity].apply(lambda col: col.first_valid_index()).sort_values()

Russell 3000                                1990-01-02
Russell 1000                                1990-01-02
SPDR S&P 500 ETF                            1993-01-29
SPDR Dow Jones Industrial Average ETF       1998-01-20
Energy Select Sector SPDR                   1998-12-22
Materials Select Sector SPDR                1998-12-22
Utilities Select Sector SPDR                1998-12-22
Industrial Select Sector SPDR               1998-12-22
Technology Select Sector SPDR               1998-12-22
Health Care Select Sector SPDR              1998-12-22
Consumer Discretionary Select Sector SPDR   1998-12-22
Consumer Staples Select Sector SPDR         1998-12-22
Financial Select Sector SPDR                1998-12-22
Invesco QQQ Trust                           1999-03-10
iShares Russell 1000 ETF                    2000-05-19
iShares Russell 1000 Value ETF              2000-05-26
iShares Russell 2000 ETF                    2000-05-26
iShares S&P 500 Growth ETF                  2000-05-26
iShares Ru

In [90]:
universe_close_prices.apply(lambda col: col.first_valid_index()).sort_values()

IVV - iShares Core S&P 500 ETF                        2000-05-19
IJH - iShares Core S&P Mid-Cap ETF                    2000-05-26
IWM - iShares Russell 2000 ETF                        2000-05-26
IYR - iShares U.S. Real Estate ETF                    2000-06-19
EFA - iShares MSCI EAFE ETF                           2001-08-17
EEM - iShares MSCI Emerging Markets ETF               2003-04-11
AGG - iShares Core U.S. Aggregate Bond ETF            2003-09-26
GLD - SPDR Gold Shares                                2004-11-18
DBC - Invesco DB Commodity Index Tracking Fund        2006-02-03
HYG - iShares iBoxx $ High Yield Corporate Bond ETF   2007-04-11
SPTL - SPDR Portfolio Long Term Treasury ETF          2007-05-30
SPBO - SPDR Portfolio Corporate Bond ETF              2011-04-07
dtype: datetime64[ns]

In [91]:
commodity_close_prices.apply(lambda col: col.first_valid_index()).sort_values()

US Dollar             1990-01-01
Lumber Futures        1990-01-01
Live Cattle Futures   1990-01-01
Gold Futures          1990-01-02
Wheat Futures         1990-01-02
Corn Futures          1990-01-02
Copper                1990-01-02
Sugar                 1990-01-02
Silver Futures        1990-01-02
Soybean Futures       1990-01-02
Coffee                1990-01-02
Natural Gas           2000-08-30
Heating Oil           2000-09-01
Brent Crude Oil       2006-05-25
Aluminum Futures      2014-05-06
dtype: datetime64[ns]

---
## Single Date Optimization Function

In [92]:
def optimize_single_date(REBAL_DATE, idx, total, ASSET_NAMES, ASSET_CLASS, ALLOW_SHORT, GROSS_EXPOSURE, N_DAYS,
                          N_SIMULATIONS, RETRAIN_KMRF, RANDOM_SEED, USE_NEAREST_MODEL=False):
    print(f"\n=== Rebalance {idx+1}/{total}: {REBAL_DATE} ===")
    
    # Initialize portfolio
    portfolio = PortfolioOptimizerInputs(
        asset_names=ASSET_NAMES,
        asset_class=ASSET_CLASS,
        end_date=REBAL_DATE,
        n_days=N_DAYS,
        n_simulations=N_SIMULATIONS,
        retrain_kmrf=RETRAIN_KMRF,
        random_seed=RANDOM_SEED,
        use_nearest_model=USE_NEAREST_MODEL
    )
    
    print(f"\n✓ PortfolioOptimizerInputs initialized")
    print(f"  - {len(portfolio.asset_names)} assets")
    print(f"  - {len(portfolio.simulators)} simulators created")
    
    # Run copula simulation
    print(f"\nRunning multi-asset copula simulation...")
    simulations = portfolio.simulate_all_assets(verbose=False, use_copula=True)
    
    print(f"\n✓ Copula simulation complete")
    print(f"  - Generated simulations for {len(simulations)} assets")
    print(f"  - Each simulation shape: {list(simulations.values())[0].shape}")
    
    # Calculate optimization inputs from copula simulations
    print("\nCalculating portfolio optimization inputs...")
    _opt_inputs = portfolio.get_optimization_inputs(
        method='path_covariance',
        annualize=True,
    )
    
    if ALLOW_SHORT == False:
        GROSS_EXPOSURE = None
    # Create optimizer
    optimizer = PortfolioOptimizer.from_portfolio_inputs(
        portfolio,
        objective='max_sharpe',
        allow_short=ALLOW_SHORT,
        gross_exposure=GROSS_EXPOSURE,
    )
    
    # Optimize
    print("\nOptimizing portfolio...")
    _opt_weights = optimizer.optimize(verbose=False)
    
    print(f"✓ Completed {REBAL_DATE}")
    
    return (REBAL_DATE, optimizer)


---
## Backtest Inputs

In [93]:
i = -12
len(all_rebal_dates[i:])
print("First Rebalance Date:", all_rebal_dates[i])
print("Last Rebalance Date:", all_rebal_dates[-1])

First Rebalance Date: 20241101
Last Rebalance Date: 20251007


#### Backtest Date Generator

In [94]:
def generate_rebalance_dates(
    start_date: str,
    end_date: str,
    frequency: str = 'monthly',
    custom_interval: Optional[int] = None,
    price_data: Optional[pd.DataFrame] = None
) -> list:
    """
    Generate rebalance dates based on start/end dates and frequency.
    
    This function intelligently generates rebalance dates without requiring
    alignment with KAMA-MSR model training dates. It works with:
    - Any start/end date combination
    - Multiple frequency options
    - Actual trading days from price data
    
    Parameters
    ----------
    start_date : str
        Start date in 'YYYY-MM-DD' or 'YYYYMMDD' format
    end_date : str
        End date in 'YYYY-MM-DD' or 'YYYYMMDD' format
    frequency : str, default='monthly'
        Rebalancing frequency. Options:
        - 'daily': Every trading day
        - 'weekly': Every 5 trading days
        - 'biweekly': Every 10 trading days
        - 'monthly': Every 21 trading days
        - 'quarterly': Every 63 trading days
        - 'custom': Use custom_interval parameter
    custom_interval : int, optional
        Number of trading days between rebalances (used with frequency='custom')
    price_data : pd.DataFrame, optional
        DataFrame with DatetimeIndex of trading days. If None, uses SPY data.
        
    Returns
    -------
    list
        List of rebalance dates in 'YYYYMMDD' format
        
    Examples
    --------
    >>> # Monthly rebalancing for 2024
    >>> dates = generate_rebalance_dates('2024-01-01', '2024-12-31', 'monthly')
    
    >>> # Weekly rebalancing for Q4 2024
    >>> dates = generate_rebalance_dates('2024-10-01', '2024-12-31', 'weekly')
    
    >>> # Every 3 days using custom interval
    >>> dates = generate_rebalance_dates('2024-10-01', '2024-12-31', 'custom', custom_interval=3)
    """
    # Use SPY data if none provided
    if price_data is None:
        price_data = etf_close_prices['SPDR S&P 500 ETF'].to_frame().dropna()
    
    # Parse dates
    start_ts = pd.Timestamp(start_date)
    end_ts = pd.Timestamp(end_date)
    
    # Get all trading days in range
    all_trading_days = price_data.loc[start_ts:end_ts].index
    
    if len(all_trading_days) == 0:
        raise ValueError(f"No trading days found between {start_date} and {end_date}")
    
    # Map frequency to interval (in trading days)
    frequency_map = {
        'daily': 1,
        'weekly': 5,
        'biweekly': 10,
        'monthly': 21,
        'quarterly': 63,
        'custom': custom_interval
    }
    
    if frequency not in frequency_map:
        raise ValueError(
            f"Unknown frequency: {frequency}. "
            f"Choose from: {list(frequency_map.keys())}"
        )
    
    interval = frequency_map[frequency]
    
    if interval is None:
        raise ValueError("Must provide custom_interval when using frequency='custom'")
    
    # Generate rebalance dates at specified interval
    # Only include rebalances that occur BEFORE the end_date
    # The end_date is the testing period end, not necessarily a rebalance date
    rebal_dates = all_trading_days[::interval]
    
    # Filter out any rebalance dates that are >= end_ts
    # We want rebalances that happen before the end of the testing period
    rebal_dates = rebal_dates[rebal_dates < end_ts]
    
    # Convert to YYYYMMDD format
    rebal_dates_str = rebal_dates.map(lambda dte: dte.strftime('%Y%m%d')).to_list()
    
    return rebal_dates_str

def summarize_backtest_config(
    rebal_dates: list,
    asset_names: list,
    asset_class: str,
    n_days: int,
    n_simulations: int,
    use_nearest_model: bool,
    retrain_kmrf: bool
):
    """
    Print a summary of backtest configuration.
    
    Shows key information about the backtest setup including:
    - Number of rebalances and date range
    - Portfolio composition
    - Model usage strategy
    - Computational requirements
    """
    print("=" * 80)
    print("BACKTEST CONFIGURATION SUMMARY")
    print("=" * 80)
    
    print(f"\n📅 REBALANCING SCHEDULE:")
    print(f"  • Total rebalances: {len(rebal_dates)}")
    print(f"  • First rebalance:  {rebal_dates[0]} ({pd.Timestamp(rebal_dates[0]).strftime('%Y-%m-%d')})")
    print(f"  • Last rebalance:   {rebal_dates[-1]} ({pd.Timestamp(rebal_dates[-1]).strftime('%Y-%m-%d')})")
    
    # Calculate average interval
    dates_ts = [pd.Timestamp(d) for d in rebal_dates]
    if len(dates_ts) > 1:
        intervals = [(dates_ts[i+1] - dates_ts[i]).days for i in range(len(dates_ts)-1)]
        avg_interval = sum(intervals) / len(intervals)
        print(f"  • Average interval: {avg_interval:.1f} days")
    
    print(f"\n📊 PORTFOLIO:")
    print(f"  • Asset class:      {asset_class}")
    print(f"  • Number of assets: {len(asset_names)}")
    print(f"  • Forecast horizon: {n_days} days")
    print(f"  • Simulations:      {n_simulations:,}")
    
    print(f"\n🤖 MODEL STRATEGY:")
    print(f"  • Use nearest model: {use_nearest_model}")
    if use_nearest_model:
        print(f"    → Will use most recent monthly KAMA-MSR model for each rebalance")
        print(f"    → No model retraining needed for intermediate dates")
    else:
        print(f"    → Requires exact KAMA-MSR model match for each date")
    
    print(f"  • Retrain KMRF:     {retrain_kmrf}")
    if retrain_kmrf:
        print(f"    → Will train new KMRF models (slower, ~35-40 min per rebalance)")
    else:
        print(f"    → Will use pre-trained KMRF models (faster, ~20-30 min per rebalance)")
    
    print(f"\n⚡ ESTIMATED RUNTIME:")
    if retrain_kmrf:
        est_time_per_rebal = 38  # minutes
    else:
        est_time_per_rebal = 25  # minutes
    
    total_est_minutes = len(rebal_dates) * est_time_per_rebal
    total_est_hours = total_est_minutes / 60
    
    print(f"  • Per rebalance:    ~{est_time_per_rebal} minutes")
    print(f"  • Total (serial):   ~{total_est_hours:.1f} hours")
    

    # With parallelization    print("\n" + "=" * 80)

    n_cpus = min(multiprocessing.cpu_count(), 6)    

    parallel_hours = total_est_hours / n_cpus    
    print(f"  • Total ({n_cpus} CPUs):   ~{parallel_hours:.1f} hours")

In [95]:
# ============================================================================
# BACKTEST CONFIGURATION
# ============================================================================

# Portfolio Configuration
ASSET_NAMES = asset_names_df['us_equity'][4:13].tolist()
ASSET_CLASS = 'us_equity'

# Strategy Configuration
ALLOW_SHORT = True
GROSS_EXPOSURE = 1.5
N_DAYS = 21  # Forecast horizon
N_SIMULATIONS = 1000
RANDOM_SEED = 42

# Model Configuration
RETRAIN_KMRF = True
USE_NEAREST_MODEL = True  # Enable flexible date handling

# Date Range Configuration
# Option 1: Use specific start/end dates (any dates, don't need to align with models!)
BACKTEST_START = '2022-12-30'  # Can be any date
BACKTEST_END = '2024-01-01'    # Can be any date
REBAL_FREQUENCY = 'monthly'     # 'daily', 'weekly', 'biweekly', 'monthly', 'quarterly', 'custom'
CUSTOM_INTERVAL = 3         # Only used if REBAL_FREQUENCY='custom' (e.g., 7 for every 7 days)

# Option 2: Or use the old approach with all_rebal_dates
# REBAL_DATES = all_rebal_dates[-12:]  # Last 12 monthly rebalances

# Generate rebalance dates
REBAL_DATES = generate_rebalance_dates(
    start_date=BACKTEST_START,
    end_date=BACKTEST_END,
    frequency=REBAL_FREQUENCY,
    custom_interval=CUSTOM_INTERVAL
)

# Parallel Processing
USE_N_CPUS = 4  # Set to desired number of CPUs (or -1 to use all available)

# ============================================================================
# SUMMARY
# ============================================================================
summarize_backtest_config(
    rebal_dates=REBAL_DATES,
    asset_names=ASSET_NAMES,
    asset_class=ASSET_CLASS,
    n_days=N_DAYS,
    n_simulations=N_SIMULATIONS,
    use_nearest_model=USE_NEAREST_MODEL,
    retrain_kmrf=RETRAIN_KMRF
)

expos = GROSS_EXPOSURE if ALLOW_SHORT else 1
freq = REBAL_FREQUENCY if REBAL_FREQUENCY != 'custom' else f'custom-{CUSTOM_INTERVAL}'
BACKTEST_SAVE_NAME = "sectors" + f"_gross-{expos}_{freq}_{BACKTEST_START.replace('-', '')}-{BACKTEST_END.replace('-', '')}.pkl"
print("\n" + BACKTEST_SAVE_NAME)

BACKTEST CONFIGURATION SUMMARY

📅 REBALANCING SCHEDULE:
  • Total rebalances: 12
  • First rebalance:  20221230 (2022-12-30)
  • Last rebalance:   20231201 (2023-12-01)
  • Average interval: 30.5 days

📊 PORTFOLIO:
  • Asset class:      us_equity
  • Number of assets: 9
  • Forecast horizon: 21 days
  • Simulations:      1,000

🤖 MODEL STRATEGY:
  • Use nearest model: True
    → Will use most recent monthly KAMA-MSR model for each rebalance
    → No model retraining needed for intermediate dates
  • Retrain KMRF:     True
    → Will train new KMRF models (slower, ~35-40 min per rebalance)

⚡ ESTIMATED RUNTIME:
  • Per rebalance:    ~38 minutes
  • Total (serial):   ~7.6 hours
  • Total (6 CPUs):   ~1.3 hours

sectors_gross-1.5_monthly_20221230-20240101.pkl


#### Model Date Alignment Checker

In [96]:
def show_model_alignment(rebal_dates: list, asset_class: str, max_display: int = 10):
    """
    Show which KAMA-MSR model will be used for each rebalance date.
    
    This helps visualize how the nearest-model strategy works and
    how "stale" the models might be for high-frequency rebalancing.
    """
    from pathlib import Path
    
    models_path = Path('saved_models/KAMA_MSR') / asset_class
    
    # Get all available model dates
    available_models = sorted([
        d.name for d in models_path.iterdir() 
        if d.is_dir() and d.name.isdigit() and len(d.name) == 8
    ])
    
    print("=" * 80)
    print(f"MODEL ALIGNMENT CHECK: {asset_class}")
    print("=" * 80)
    print(f"\nAvailable KAMA-MSR models: {len(available_models)}")
    print(f"Date range: {available_models[0]} to {available_models[-1]}")
    print("\n" + "-" * 80)
    
    # Show first N rebalances
    display_dates = rebal_dates[:max_display]
    
    print(f"\nShowing first {len(display_dates)} of {len(rebal_dates)} rebalances:")
    print(f"{'Rebalance Date':<20} {'Model Used':<20} {'Staleness':<15} {'Status'}")
    print("-" * 80)
    
    model_usage_count = {}
    
    for rebal_date in display_dates:
        rebal_ts = pd.Timestamp(rebal_date)
        
        # Find nearest model
        nearest_model = None
        for model_date in reversed(available_models):
            if pd.Timestamp(model_date) <= rebal_ts:
                nearest_model = model_date
                break
        
        if nearest_model:
            days_stale = (rebal_ts - pd.Timestamp(nearest_model)).days
            exact_match = (rebal_date == nearest_model)
            status = "✓ Exact" if exact_match else f"↻ {days_stale}d old"
            
            model_usage_count[nearest_model] = model_usage_count.get(nearest_model, 0) + 1
            
            print(f"{rebal_date:<20} {nearest_model:<20} {days_stale:>3} days      {status}")
        else:
            print(f"{rebal_date:<20} {'NOT FOUND':<20} {'---':<15} ✗ Error")
    
    if len(rebal_dates) > max_display:
        print(f"... and {len(rebal_dates) - max_display} more rebalances")
    
    print("\n" + "-" * 80)
    print(f"\nMODEL REUSE SUMMARY:")
    print(f"Unique models used: {len(model_usage_count)}")
    for model_date, count in sorted(model_usage_count.items()):
        print(f"  • {model_date}: used {count} time(s)")
    
    # Calculate efficiency gain
    if len(model_usage_count) < len(rebal_dates):
        efficiency = (1 - len(model_usage_count) / len(rebal_dates)) * 100
        print(f"\n💡 Model reuse efficiency: {efficiency:.1f}%")
        print(f"   (Avoiding {len(rebal_dates) - len(model_usage_count)} unnecessary model trainings)")
    
    print("=" * 80)


# Show model alignment for current configuration
show_model_alignment(REBAL_DATES, ASSET_CLASS, max_display=15)


MODEL ALIGNMENT CHECK: us_equity

Available KAMA-MSR models: 82
Date range: 20181231 to 20251007

--------------------------------------------------------------------------------

Showing first 12 of 12 rebalances:
Rebalance Date       Model Used           Staleness       Status
--------------------------------------------------------------------------------
20221230             20221230               0 days      ✓ Exact
20230201             20230201               0 days      ✓ Exact
20230303             20230303               0 days      ✓ Exact
20230403             20230403               0 days      ✓ Exact
20230503             20230503               0 days      ✓ Exact
20230602             20230602               0 days      ✓ Exact
20230705             20230705               0 days      ✓ Exact
20230803             20230803               0 days      ✓ Exact
20230901             20230901               0 days      ✓ Exact
20231003             20231003               0 days      ✓ Exact

---
## Parallelized Backtest Loop

In [97]:
print(f"\n{'='*80}")
print(f"PARALLELIZED BACKTEST: {len(REBAL_DATES)} rebalance dates using {USE_N_CPUS} CPUs")
print(f"{'='*80}")

# Run parallel optimization
results = Parallel(n_jobs=USE_N_CPUS, verbose=10)(
    delayed(optimize_single_date)(
        REBAL_DATE, idx, len(REBAL_DATES), 
        ASSET_NAMES, ASSET_CLASS, ALLOW_SHORT, GROSS_EXPOSURE, N_DAYS, N_SIMULATIONS, RETRAIN_KMRF, RANDOM_SEED,
        USE_NEAREST_MODEL  # Pass the USE_NEAREST_MODEL parameter
    )
    for idx, REBAL_DATE in enumerate(REBAL_DATES)
)

# Convert results to dictionary
BACKTEST_OPTIMIZERS = dict(results)


PARALLELIZED BACKTEST: 12 rebalance dates using 4 CPUs


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.



=== Rebalance 4/12: 20230403 ===

=== Rebalance 1/12: 20221230 ===
Using nearest model date: 20221230 for rebalance date: 20221230 (0 days stale)

=== Rebalance 2/12: 20230201 ===
Using nearest model date: 20230403 for rebalance date: 20230403 (0 days stale)

=== Rebalance 3/12: 20230303 ===
Using nearest model date: 20230201 for rebalance date: 20230201 (0 days stale)
Using nearest model date: 20230303 for rebalance date: 20230303 (0 days stale)

✓ PortfolioOptimizerInputs initialized
  - 9 assets
  - 0 simulators created
✓ PortfolioOptimizerInputs initialized

  - 9 assets

Running multi-asset copula simulation...
  - 0 simulators created

Running multi-asset copula simulation...

✓ PortfolioOptimizerInputs initialized
  - 9 assets
  - 0 simulators created

Running multi-asset copula simulation...

✓ PortfolioOptimizerInputs initialized
  - 9 assets
  - 0 simulators created

Running multi-asset copula simulation...
KMRF model initialized
  Asset: Energy Select Sector SPDR
  Asset cl

KeyboardInterrupt: 

---
## Save Backtest

In [ ]:
BACKTEST_SAVE_NAME

'sectors_gross-1.5_monthly_20240103-20241031.pkl'

In [ ]:
with open(f'backtests/{BACKTEST_SAVE_NAME}', 'wb') as f:
    pickle.dump(BACKTEST_OPTIMIZERS, f)